# U5-01 · Regresión regularizada y regresión logística

*Machine Learning e Inteligencia Artificial (IFCD0174) · Unidad 5, apartados 5.1 a 5.3*

Ejecuta cada celda con **Mayús + Intro**. Los conjuntos de datos vienen incluidos en scikit-learn; no hace falta descargar nada.

## 1. Regresión lineal y modelo de referencia (ejemplo 5.1)

Datos de diabetes: 442 pacientes, 10 variables y la progresión de la enfermedad un año después (Efron et al., 2004).

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, root_mean_squared_error

X, y = load_diabetes(return_X_y=True, as_frame=True)
X_ent, X_pru, y_ent, y_pru = train_test_split(X, y, test_size=0.25, random_state=0)
modelo = LinearRegression().fit(X_ent, y_ent)
for nombre, (a, b) in {"entrenamiento": (X_ent, y_ent), "prueba": (X_pru, y_pru)}.items():
    p = modelo.predict(a)
    print(f"{nombre:>13}: R2 = {r2_score(b, p):.3f}  RMSE = {root_mean_squared_error(b, p):.1f}")
ref = np.full(len(y_pru), y_ent.mean())
print(f"   referencia: RMSE = {root_mean_squared_error(y_pru, ref):.1f}  (predecir siempre la media)")

## 2. Ridge y lasso con validación cruzada (ejemplo 5.2)

In [ ]:
from sklearn.linear_model import LassoCV, RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

lasso = make_pipeline(StandardScaler(), LassoCV(cv=5)).fit(X_ent, y_ent)
ridge = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 61))).fit(X_ent, y_ent)
print("Lasso: lambda =", round(lasso[-1].alpha_, 2), "· anula:", list(X.columns[lasso[-1].coef_ == 0]),
      "· RMSE prueba =", round(root_mean_squared_error(y_pru, lasso.predict(X_pru)), 1))
print("Ridge: lambda =", round(ridge[-1].alpha_, 2),
      "· RMSE prueba =", round(root_mean_squared_error(y_pru, ridge.predict(X_pru)), 1))
pd.DataFrame({"lasso": lasso[-1].coef_, "ridge": ridge[-1].coef_}, index=X.columns).round(1)

### Trayectorias del lasso (figura 5.2)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.linear_model import lasso_path

Z = StandardScaler().fit_transform(X_ent)
alphas, coefs, _ = lasso_path(Z, y_ent - y_ent.mean(), alphas=np.logspace(-1, 1.7, 60))
fig, ax = plt.subplots(figsize=(7, 4))
for nombre, c in zip(X.columns, coefs):
    ax.plot(np.log10(alphas), c, label=nombre)
ax.axvline(np.log10(lasso[-1].alpha_), ls=":", color="gray")
ax.set_xlabel("log10(lambda)"); ax.set_ylabel("coeficiente"); ax.legend(ncol=2, fontsize=8)
plt.show()

**Ejercicio.** Sustituye `LassoCV` por `ElasticNetCV(l1_ratio=[.2, .5, .8], cv=5)`. ¿Cuántos coeficientes anula ahora? ¿Cambia el error de prueba?

## 3. Regresión logística y umbral (ejemplo 5.3)

Cáncer de mama de Wisconsin: 569 tumores, 30 variables (Street et al., 1993). Codificamos 1 = maligno.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix

Xc, yc = load_breast_cancer(return_X_y=True, as_frame=True)
yc = 1 - yc
Xc_ent, Xc_pru, yc_ent, yc_pru = train_test_split(Xc, yc, test_size=0.25, stratify=yc, random_state=0)
logit = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)).fit(Xc_ent, yc_ent)
prob = logit.predict_proba(Xc_pru)[:, 1]
filas = []
for u in [0.5, 0.3, 0.1]:
    vn, fp, fn, vp = confusion_matrix(yc_pru, (prob >= u).astype(int)).ravel()
    filas.append({"umbral": u, "detectados": vp, "no detectados": fn, "falsas alarmas": fp})
pd.DataFrame(filas)

### ¿Están bien calibradas las probabilidades?

In [ ]:
from sklearn.calibration import calibration_curve
frac, media = calibration_curve(yc_pru, prob, n_bins=5)
plt.plot(media, frac, "o-", label="regresión logística")
plt.plot([0, 1], [0, 1], ":", color="gray", label="calibración perfecta")
plt.xlabel("probabilidad predicha"); plt.ylabel("proporción real de malignos"); plt.legend(); plt.show()

**Ejercicio.** Busca el umbral más alto que detecta los 53 tumores malignos de prueba. ¿Cuántas falsas alarmas produce?

> Cox, D. R. (1958). The Regression Analysis of Binary Sequences. *JRSS B*, 20(2), 215–242. https://doi.org/10.1111/j.2517-6161.1958.tb00292.x
>
> Tibshirani, R. (1996). Regression Shrinkage and Selection via the Lasso. *JRSS B*, 58(1), 267–288. https://doi.org/10.1111/j.2517-6161.1996.tb02080.x